# Chapter 2 — Hashing

**DS/MLE perspective first:** `value_counts()`, `groupby`, hash joins, frequency encoding — all hash maps under the hood. This chapter shows the algorithms, then the two LeetCode problems per pattern that test them from scratch.

Two patterns, same order as `notes.md`:
1. **Hash Maps & Frequency Counting** → `Counter`, `value_counts()`, `groupby` — LC 1, LC 242
2. **Hashing + Sliding Window** → rolling n-gram extraction, window anagram search — LC 438, LC 567

Each section: DS/ML version first → mechanism → LeetCode problem → side-by-side implementations → step-by-step trace.

In [11]:
import numpy as np
import pandas as pd
from collections import Counter, defaultdict

# ── Trace helpers ──────────────────────────────────────────────────────────────

def fmt_dict(d):
    """Format a dict/Counter compactly for trace output."""
    return "{" + ", ".join(f"{k!r}:{v}" for k, v in sorted(d.items(), key=lambda x: str(x[0]))) + "}"

def show_freq(d, label="freq"):
    """Print a frequency map with sorted keys."""
    print(f"  {label}: {fmt_dict(d)}")

---
## Part 1 — Hash Maps & Frequency Counting

### What you already use: `Counter` and `value_counts()`

Frequency maps are the most common hash map pattern in DS work — counting how often each value appears, then acting on those counts.

In [12]:
# Simulated clickstream: user action types
actions = ['click', 'view', 'click', 'purchase', 'click', 'view', 'view', 'click']

# pandas: the tool you use every day
counts_pd = pd.Series(actions).value_counts()
print("pd.value_counts():")
print(counts_pd.to_string())
print()

# Counter: pure Python, same idea
counts_ct = Counter(actions)
print(f"Counter: {dict(counts_ct)}")
print(f"Most common 2: {counts_ct.most_common(2)}")
print()

# Both are hash maps: key → count
# Under the hood, value_counts() builds the same dict Counter does,
# then sorts by frequency.

pd.value_counts():
click       4
view        3
purchase    1

Counter: {'click': 4, 'view': 3, 'purchase': 1}
Most common 2: [('click', 4), ('view', 3)]



### Under the hood: O(1) lookup via hashing

For each element, Python computes `hash(element)` to determine which bucket it goes in. Lookup is O(1) average — no scanning needed. The hash map is the reason `x in set(list)` is O(1) while `x in list` is O(n).

```
'click' → hash('click') → bucket 7 → count: 4
'view'  → hash('view')  → bucket 2 → count: 3
```

Two key patterns:
- **Complement lookup**: store what you've seen, check if the complement is already there (LC 1)
- **Frequency comparison**: count both things, compare the maps (LC 242)

---
### LC 1 — [Two Sum](https://leetcode.com/problems/two-sum/)

> Given an array `nums` and a `target`, return the indices of the two numbers that add up to `target`. Exactly one solution; can't use the same element twice.
>
> Example: `nums = [2, 7, 11, 15], target = 9` → `[0, 1]`

**DS parallel:** Checking if any two feature values in a column sum to a threshold — e.g., finding a complementary pair in a sorted price list. In pandas: `df.merge(df, how='inner')` where `val1 + val2 == target`, but that's O(n²). The hash map does it in O(n).

**Key insight:** For each `num`, check if `target - num` is already in a seen dict. One pass — by the time we need the complement, we've already stored it.

In [13]:
# ── NUMPY APPROACH (O(n²) broadcast — readable but doesn't scale) ─────────────
def two_sum_numpy(nums, target):
    arr = np.array(nums)
    # Outer product of sums: arr[i] + arr[j] for all i, j
    sums = arr[:, None] + arr[None, :]      # O(n²) space
    rows, cols = np.where((sums == target) & (np.arange(len(arr))[:, None] != np.arange(len(arr))))
    if len(rows): return [int(rows[0]), int(cols[0])]
    return []

# ── HASH MAP (the interview answer) ───────────────────────────────────────────
# O(n) time, O(n) space. One pass.
def twoSum(nums, target):
    seen = {}          # value → index
    for i, num in enumerate(nums):
        complement = target - num
        if complement in seen:
            return [seen[complement], i]
        seen[num] = i
    return []

cases = [
    ([2, 7, 11, 15], 9,  [0, 1]),
    ([3, 2, 4],      6,  [1, 2]),
    ([3, 3],         6,  [0, 1]),
]
print(f"{'nums':<20} {'target':>6}  {'numpy':>8}  {'hash':>8}  {'expected':>10}")
print("-" * 60)
for nums, tgt, expected in cases:
    np_ans   = two_sum_numpy(nums, tgt)
    hash_ans = twoSum(nums, tgt)
    match = "✓" if hash_ans == expected else "✗"
    print(f"{str(nums):<20} {tgt:>6}  {str(np_ans):>8}  {str(hash_ans):>8}  {str(expected):>10}  {match}")

nums                 target     numpy      hash    expected
------------------------------------------------------------
[2, 7, 11, 15]            9    [0, 1]    [0, 1]      [0, 1]  ✓
[3, 2, 4]                 6    [1, 2]    [1, 2]      [1, 2]  ✓
[3, 3]                    6    [0, 1]    [0, 1]      [0, 1]  ✓


In [14]:
# ── Step-by-step trace — shows seen dict evolving at every step ───────────────
nums, target = [2, 7, 11, 15], 9
print(f"nums = {nums},  target = {target}")
print(f"Looking for pairs where nums[i] + nums[j] == {target}")
print()
print(f"  {'i':>3}  {'num':>5}  {'complement':>11}  {'in seen?':>10}  {'action':<30}  seen after")
print("  " + "-" * 75)

seen = {}
for i, num in enumerate(nums):
    complement = target - num
    in_seen = complement in seen
    if in_seen:
        action = f"FOUND! return [{seen[complement]}, {i}]"
    else:
        action = f"store seen[{num}] = {i}"
        seen[num] = i
    seen_str = fmt_dict(seen)
    print(f"  {i:>3}  {num:>5}  {complement:>11}  {str(in_seen):>10}  {action:<30}  {seen_str}")
    if in_seen: break

nums = [2, 7, 11, 15],  target = 9
Looking for pairs where nums[i] + nums[j] == 9

    i    num   complement    in seen?  action                          seen after
  ---------------------------------------------------------------------------
    0      2            7       False  store seen[2] = 0               {2:0}
    1      7            2        True  FOUND! return [0, 1]            {2:0}


---
### LC 242 — [Valid Anagram](https://leetcode.com/problems/valid-anagram/)

> Given strings `s` and `t`, return `True` if `t` is an anagram of `s` (same characters, same frequencies, different order).
>
> Example: `s = "anagram", t = "nagaram"` → `True`

**DS parallel:** Checking if two categorical columns have the same value distribution — `s.value_counts().equals(t.value_counts())`. In NLP, anagram detection is character-level bag-of-words comparison.

**Key insight:** Two strings are anagrams iff their character frequency maps are identical. `Counter(s) == Counter(t)` does this in O(n). The raw version builds the map manually to show the mechanism.

In [15]:
# ── PYTHONIC (Counter comparison — what you'd write in a DS script) ───────────
def isAnagram_pythonic(s, t):
    return Counter(s) == Counter(t)

# ── RAW PYTHON (the interview answer — shows the mechanism) ───────────────────
# O(n) time, O(1) space (at most 26 lowercase letters in freq map)
def isAnagram(s, t):
    if len(s) != len(t):
        return False
    freq = {}
    for c in s:
        freq[c] = freq.get(c, 0) + 1   # count s
    for c in t:
        freq[c] = freq.get(c, 0) - 1   # decrement for t
        if freq[c] < 0:
            return False                # t has a char s doesn't
    return True

cases = [
    ("anagram", "nagaram", True),
    ("rat",     "car",     False),
    ("ab",      "a",       False),
    ("aab",     "bba",     False),
]
print(f"{'s':<12} {'t':<12} {'pythonic':>10}  {'raw':>10}  {'expected':>10}")
print("-" * 58)
for s, t, expected in cases:
    p = isAnagram_pythonic(s, t)
    r = isAnagram(s, t)
    match = "✓" if p == r == expected else "✗"
    print(f"{s!r:<12} {t!r:<12} {str(p):>10}  {str(r):>10}  {str(expected):>10}  {match}")

s            t              pythonic         raw    expected
----------------------------------------------------------
'anagram'    'nagaram'          True        True        True  ✓
'rat'        'car'             False       False       False  ✓
'ab'         'a'               False       False       False  ✓
'aab'        'bba'             False       False       False  ✓


In [16]:
# ── Step-by-step trace — freq map building and decrement phases ───────────────
s, t = "anagram", "nagaram"
print(f"s = {s!r},  t = {t!r}")
print()
print("Phase 1: count characters in s (increment)")
freq = {}
for i, c in enumerate(s):
    freq[c] = freq.get(c, 0) + 1
    print(f"  s[{i}]={c!r}:  freq = {fmt_dict(freq)}")

print()
print("Phase 2: decrement for each character in t")
ok = True
for i, c in enumerate(t):
    freq[c] = freq.get(c, 0) - 1
    status = "ok" if freq[c] >= 0 else "FAIL (t has excess)"
    print(f"  t[{i}]={c!r}:  freq[{c!r}] → {freq[c]}   {status}   freq = {fmt_dict(freq)}")
    if freq[c] < 0:
        ok = False
        break

print()
all_zero = all(v == 0 for v in freq.values())
print(f"Final freq map: {fmt_dict(freq)}")
print(f"All zeros? {all_zero}  →  isAnagram = {ok and all_zero}")

s = 'anagram',  t = 'nagaram'

Phase 1: count characters in s (increment)
  s[0]='a':  freq = {'a':1}
  s[1]='n':  freq = {'a':1, 'n':1}
  s[2]='a':  freq = {'a':2, 'n':1}
  s[3]='g':  freq = {'a':2, 'g':1, 'n':1}
  s[4]='r':  freq = {'a':2, 'g':1, 'n':1, 'r':1}
  s[5]='a':  freq = {'a':3, 'g':1, 'n':1, 'r':1}
  s[6]='m':  freq = {'a':3, 'g':1, 'm':1, 'n':1, 'r':1}

Phase 2: decrement for each character in t
  t[0]='n':  freq['n'] → 0   ok   freq = {'a':3, 'g':1, 'm':1, 'n':0, 'r':1}
  t[1]='a':  freq['a'] → 2   ok   freq = {'a':2, 'g':1, 'm':1, 'n':0, 'r':1}
  t[2]='g':  freq['g'] → 0   ok   freq = {'a':2, 'g':0, 'm':1, 'n':0, 'r':1}
  t[3]='a':  freq['a'] → 1   ok   freq = {'a':1, 'g':0, 'm':1, 'n':0, 'r':1}
  t[4]='r':  freq['r'] → 0   ok   freq = {'a':1, 'g':0, 'm':1, 'n':0, 'r':0}
  t[5]='a':  freq['a'] → 0   ok   freq = {'a':0, 'g':0, 'm':1, 'n':0, 'r':0}
  t[6]='m':  freq['m'] → 0   ok   freq = {'a':0, 'g':0, 'm':0, 'n':0, 'r':0}

Final freq map: {'a':0, 'g':0, 'm':0, 'n':0, 'r'

---
## Part 2 — Hashing + Sliding Window

### What you already use: rolling n-gram extraction

Fixed-size sliding windows over text — scanning for repeated patterns, extracting n-grams, checking if a substring matches a pattern. The key: maintain the frequency map incrementally rather than rebuilding it from scratch each step.

In [17]:
# Rolling bigram (2-gram) extraction from a log stream
log = "error warn info error error warn"
words = log.split()
k = 2  # bigram window

# pandas approach: rolling over a string Series is awkward — easier to do manually
bigrams = [tuple(words[i:i+k]) for i in range(len(words) - k + 1)]
bigram_counts = Counter(bigrams)
print("Bigram counts in log:")
for bg, cnt in bigram_counts.most_common():
    print(f"  {' '.join(bg)!r}: {cnt}")
print()
print("The pattern below shows how to maintain the frequency map incrementally")
print("instead of rebuilding it for each window position — same idea as pd.rolling().")

Bigram counts in log:
  'error warn': 2
  'warn info': 1
  'info error': 1
  'error error': 1

The pattern below shows how to maintain the frequency map incrementally
instead of rebuilding it for each window position — same idea as pd.rolling().


### Under the hood: incremental freq map maintenance

For a fixed window of size `k`: when sliding right by 1, **add** the new right character and **remove** the leftmost character. This keeps the freq map current in O(1) per step instead of O(k) for a full rebuild.

```
s = "cbaebabacd",  p = "abc"
window k=3: [c,b,a] → freq={c:1,b:1,a:1} == p_freq → match!
slide:      [b,a,e] → add 'e', remove 'c' → freq={b:1,a:1,e:1} ≠ p_freq
slide:      [a,e,b] → add 'b', remove 'b' → freq={a:1,e:1,b:1} ≠ p_freq
...
```

---
### LC 438 — [Find All Anagrams in a String](https://leetcode.com/problems/find-all-anagrams-in-a-string/)

> Given strings `s` and `p`, return all start indices in `s` where a substring of length `len(p)` is an anagram of `p`.
>
> Example: `s = "cbaebabacd", p = "abc"` → `[0, 6]`

**DS parallel:** Finding all positions in a token sequence that match a target pattern (n-gram matching). In NLP feature extraction, this is scanning a document for all windows containing exactly the target vocabulary set.

**Key insight:** Fixed window of `len(p)`. Maintain `window_freq` incrementally — add right char, remove leftmost char. Compare to `p_freq` at each step.

In [18]:
# ── NAIVE O(n·k) — rebuild Counter from scratch at each position ──────────────
def findAnagrams_naive(s, p):
    k = len(p)
    p_freq = Counter(p)
    return [i for i in range(len(s) - k + 1) if Counter(s[i:i+k]) == p_freq]

# ── FIXED SLIDING WINDOW (the interview answer) ───────────────────────────────
# O(n) — maintain window freq incrementally; O(1) comparison via match count.
def findAnagrams(s, p):
    k = len(p)
    if len(s) < k:
        return []

    p_freq = Counter(p)
    window = Counter(s[:k])          # initial window
    result = []

    if window == p_freq:
        result.append(0)

    for i in range(k, len(s)):
        # Add new right character
        window[s[i]] += 1
        # Remove leftmost character
        left_char = s[i - k]
        window[left_char] -= 1
        if window[left_char] == 0:
            del window[left_char]
        if window == p_freq:
            result.append(i - k + 1)

    return result

cases = [
    ("cbaebabacd", "abc", [0, 6]),
    ("abab",       "ab",  [0, 1, 2]),
    ("aa",         "bb",  []),
]
print(f"{'s':<14} {'p':<6} {'naive':>12}  {'sliding':>12}  {'expected':>12}")
print("-" * 60)
for s, p, expected in cases:
    n_ans = findAnagrams_naive(s, p)
    sw_ans = findAnagrams(s, p)
    match = "✓" if sw_ans == expected else "✗"
    print(f"{s!r:<14} {p!r:<6} {str(n_ans):>12}  {str(sw_ans):>12}  {str(expected):>12}  {match}")

s              p             naive       sliding      expected
------------------------------------------------------------
'cbaebabacd'   'abc'        [0, 6]        [0, 6]        [0, 6]  ✓
'abab'         'ab'      [0, 1, 2]     [0, 1, 2]     [0, 1, 2]  ✓
'aa'           'bb'             []            []            []  ✓


In [19]:
# ── Step-by-step trace — window freq and match status at every step ───────────
s, p = "cbaebabacd", "abc"
k = len(p)
p_freq = Counter(p)
print(f"s = {s!r},  p = {p!r},  k = {k}")
print(f"p_freq = {fmt_dict(p_freq)}")
print()
print(f"  {'i':>3}  {'window':>12}  {'added':>7}  {'removed':>9}  {'window_freq':<22}  {'match?':>7}  result")
print("  " + "-" * 82)

window = Counter(s[:k])
result = []
is_match = window == p_freq
if is_match: result.append(0)
print(f"  {'init':>3}  {s[:k]!r:>12}  {'':>7}  {'':>9}  {fmt_dict(window):<22}  {str(is_match):>7}  {result}")

for i in range(k, len(s)):
    added = s[i]
    removed = s[i - k]
    window[added] += 1
    window[removed] -= 1
    if window[removed] == 0:
        del window[removed]
    is_match = window == p_freq
    if is_match:
        result.append(i - k + 1)
    win_str = s[i - k + 1:i + 1]
    print(f"  {i:>3}  {win_str!r:>12}  +{added!r:>5}  -{removed!r:>7}  {fmt_dict(window):<22}  {str(is_match):>7}  {result}")

s = 'cbaebabacd',  p = 'abc',  k = 3
p_freq = {'a':1, 'b':1, 'c':1}

    i        window    added    removed  window_freq              match?  result
  ----------------------------------------------------------------------------------
  init         'cba'                      {'a':1, 'b':1, 'c':1}      True  [0]
    3         'bae'  +  'e'  -    'c'  {'a':1, 'b':1, 'e':1}     False  [0]
    4         'aeb'  +  'b'  -    'b'  {'a':1, 'b':1, 'e':1}     False  [0]
    5         'eba'  +  'a'  -    'a'  {'a':1, 'b':1, 'e':1}     False  [0]
    6         'bab'  +  'b'  -    'e'  {'a':1, 'b':2}            False  [0]
    7         'aba'  +  'a'  -    'b'  {'a':2, 'b':1}            False  [0]
    8         'bac'  +  'c'  -    'a'  {'a':1, 'b':1, 'c':1}      True  [0, 6]
    9         'acd'  +  'd'  -    'b'  {'a':1, 'c':1, 'd':1}     False  [0, 6]


---
### LC 567 — [Permutation in String](https://leetcode.com/problems/permutation-in-string/)

> Return `True` if any permutation of `p` is a substring of `s`.
>
> Example: `s = "eidbaooo", p = "ab"` → `True` (substring `"ba"` is a permutation of `"ab"`)

**DS parallel:** Checking whether a target n-gram pattern appears anywhere in a document — used in text feature validation and vocabulary matching.

**Key insight:** Same fixed-window pattern as LC 438 — just return `True` on first match instead of collecting all matches. Shows the pattern is reusable with minimal modification.

In [20]:
# ── PYTHONIC (any + Counter comparison) ───────────────────────────────────────
def checkInclusion_pythonic(p, s):
    k = len(p)
    p_freq = Counter(p)
    return any(Counter(s[i:i+k]) == p_freq for i in range(len(s) - k + 1))

# ── FIXED SLIDING WINDOW (the interview answer) ───────────────────────────────
# Identical structure to LC 438 — stop on first match.
def checkInclusion(p, s):
    k = len(p)
    if len(s) < k:
        return False
    p_freq = Counter(p)
    window = Counter(s[:k])
    if window == p_freq:
        return True
    for i in range(k, len(s)):
        window[s[i]] += 1
        left = s[i - k]
        window[left] -= 1
        if window[left] == 0:
            del window[left]
        if window == p_freq:
            return True
    return False

cases = [
    ("ab", "eidbaooo", True),
    ("ab", "eidboaoo", False),
    ("adc","dcda",    True),
]
print(f"{'p':<6} {'s':<14} {'pythonic':>10}  {'sliding':>10}  {'expected':>10}")
print("-" * 56)
for p_str, s_str, expected in cases:
    py_ans = checkInclusion_pythonic(p_str, s_str)
    sw_ans = checkInclusion(p_str, s_str)
    match = "✓" if sw_ans == expected else "✗"
    print(f"{p_str!r:<6} {s_str!r:<14} {str(py_ans):>10}  {str(sw_ans):>10}  {str(expected):>10}  {match}")

# DS application: check if a query vocabulary appears in a document window
print()
print("DS application — does any 3-word window contain exactly {error, warn, info}?")
target_vocab = "ewi"  # encode as chars for simplicity
doc_stream   = "eewiwewi"  # e=error, w=warn, i=info
print(f"  Target vocab:  {set('error warn info'.split())}")
print(f"  Found window:  {checkInclusion(target_vocab, doc_stream)}")

p      s                pythonic     sliding    expected
--------------------------------------------------------
'ab'   'eidbaooo'           True        True        True  ✓
'ab'   'eidboaoo'          False       False       False  ✓
'adc'  'dcda'               True        True        True  ✓

DS application — does any 3-word window contain exactly {error, warn, info}?
  Target vocab:  {'error', 'warn', 'info'}
  Found window:  True


---
## Summary

| Pattern | LeetCode | DS/ML Equivalent | Time | Space |
|---------|----------|------------------|------|-------|
| Hash map complement lookup | LC 1 Two Sum | `dict.get()` / feature join | O(n) | O(n) |
| Frequency map comparison | LC 242 Valid Anagram | `Counter(s) == Counter(t)` / `value_counts()` | O(n) | O(k) |
| Fixed window + freq map | LC 438 Find All Anagrams | Rolling n-gram extraction | O(n) | O(k) |
| Fixed window + freq map (exists) | LC 567 Permutation in String | Vocabulary membership check | O(n) | O(k) |

**The key habit:** whenever you see an `if x in something` in a loop, ask whether `something` should be a set or dict. Converting a list to a set before repeated lookups is the most common O(n²) → O(n) optimization in DS code.